In [1]:
from __future__ import annotations

import operator
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_groq import ChatGroq 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

/home/jiggra/Blog-Writting-Agent/blog-writing-agent/.blog/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_305772/3498718937.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
load_dotenv()
llmm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
)
model1 = ChatHuggingFace(llm=llmm)

model = ChatGroq(
    model="llama-3.3-70b-versatile",
)


In [3]:
from typing import List, Optional
from pydantic import BaseModel, Field

class Task(BaseModel):
    id: int = Field(..., description="Section order")

    title: str = Field(
        ...,
        description="Heading of the section"
    )

    objective: str = Field(
        ...,
        description="Goal of this section"
    )

    summary: str = Field(
        ...,
        description="Short explanation of what to write"
    )

    key_points: List[str] = Field(
        ...,
        description="Important points that must be covered"
    )

    keywords: List[str] = Field(
        default_factory=list,
        description="SEO keywords relevant to this section"
    )

    references_needed: bool = Field(
        False,
        description="Whether factual sources are required"
    )

    estimated_words: int = Field(
        300,
        description="Expected length"
    )

    dependencies: List[int] = Field(
        default_factory=list,
        description="IDs of sections that should be written first"
    )



class BlogMetadata(BaseModel):

    title: str

    audience: str

    tone: str

    purpose: str

    target_words: int

    language: str

    primary_keyword: str

    secondary_keywords: List[str]


class Plan(BaseModel):

    metadata: BlogMetadata

    tasks: List[Task]


class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = None  # keep if Tavily provides; DO NOT rely on it
    snippet: Optional[str] = None
    source: Optional[str] = None



class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)


class EvidencePack(BaseModel):
    evidence: List[EvidenceItem] = Field(default_factory=list)


In [4]:
class BlogState(TypedDict):
    topic: str

    # routing / research
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]


    plan: Optional[Plan]

    # workers
    sections: Annotated[List[tuple[int, str]], operator.add]  # (task_id, section_md)
    final: str




In [ ]:
ROUTER_SYSTEM = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false):
  Evergreen topics where correctness does not depend on recent facts (concepts, fundamentals).
- hybrid (needs_research=true):
  Mostly evergreen but needs up-to-date examples/tools/models to be useful.
- open_book (needs_research=true):
  Mostly volatile: weekly roundups, "this week", "latest", rankings, pricing, policy/regulation.

If needs_research=true:
- Output 3–10 high-signal queries.
- Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
- If user asked for "last week/this week/latest", reflect that constraint IN THE QUERIES.
"""



#Router Node

def router_node(state: BlogState) ->dict: